# Lab 3：PyTorch/torch_npu 模型迁移与训练

## 实验目标

- 理解 `torch_npu` 自动迁移的工作方式及其适用边界。
- 将标准 PyTorch CNN 训练代码迁移到昇腾 NPU。
- 使用 `autocast` 和 `GradScaler` 配置 NPU 混合精度训练。
- 完成模型测试与权重保存，并了解 NPU 多卡训练的 DDP 要求。

## 实验环境

| 项目 | 配置 |
| --- | --- |
| NPU | 单卡 Ascend 910B3（Atlas A2） |
| CANN | 9.0.0 |
| Python | 3.11.4 |
| 关键工具 | Jupyter Notebook、PyTorch、`torch_npu`、`torchvision` |

## 实验原理

基线代码通过 `torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')` 选择设备：有 CUDA 时使用 CUDA，没有 CUDA 时回落到 CPU。它用于确认模型、数据和训练循环能够正常运行，不作为固定的 GPU 性能基线。

Ascend Extension for PyTorch（`torch_npu`）把 PyTorch 接入 CANN。导入 `transfer_to_npu` 后，工具会在运行时把常见 CUDA API 映射到对应的 NPU API，映射范围内的训练代码可以继续使用 CUDA 风格接口。自动迁移无法覆盖所有第三方库和复杂调用，遇到未映射的接口时仍要手工调整。PyTorch、`torch_npu` 与 CANN 还需要使用相互兼容的版本组合。

AMP 使用 `autocast` 为不同运算选择合适的精度，`GradScaler` 通过缩放损失降低 FP16 梯度下溢的风险。昇腾 NPU 的多卡训练使用 `DistributedDataParallel`（DDP），不使用 `DataParallel`（DP）。

## 实验流程

### 1. 初始化实验环境

运行下面单元，加载 CANN 环境变量。之后再导入 PyTorch。

In [ ]:
import os, subprocess

env = subprocess.check_output(
    "bash -l -c 'source /home/developer/Ascend/ascend-toolkit/set_env.sh && env'",
    shell=True,
    text=True,
)
for line in env.splitlines():
    if "=" in line:
        os.environ.__setitem__(*line.split("=", 1))

### 2. 运行 PyTorch 基线

先运行标准 PyTorch 版本。代码会优先选择 CUDA；如果环境中没有 CUDA，则使用 CPU。后续迁移继续沿用同一套 CNN、MNIST 数据和训练循环。

#### 2.1 定义模型

CNN 包含两组卷积、激活和池化层，最后通过全连接层输出 10 个类别。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import time

# 初始化设备：优先使用 CUDA，无 CUDA 时使用 CPU
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

# 定义 CNN 模型
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.net = nn.Sequential(
            # 卷积层 1: 1通道 -> 16通道, 3x3卷积
            nn.Conv2d(in_channels=1, out_channels=16,
                      kernel_size=(3, 3), stride=(1, 1), padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),  # 28x28 -> 14x14
            
            # 卷积层 2: 16通道 -> 32通道
            nn.Conv2d(16, 32, 3, 1, 1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 14x14 -> 7x7
            
            # 全连接层
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)  # 10 个分类
        )
    
    def forward(self, x):
        return self.net(x)

print("模型定义完成")
model = CNN()
print(model)

#### 2.2 加载数据

加载 MNIST 训练集和测试集，并使用固定的均值与标准差完成归一化。

In [ ]:
# 数据预处理
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# 加载 MNIST 数据集
train_dataset = torchvision.datasets.MNIST(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.MNIST(
    root='./data', train=False, download=True, transform=transform
)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"训练集大小: {len(train_dataset)}")
print(f"测试集大小: {len(test_dataset)}")
print(f"批大小: {batch_size}")
print(f"训练批次数: {len(train_loader)}")

#### 2.3 训练基线模型

模型、损失函数和每批数据都通过 `.to(device)` 移到当前设备。设备可能是 CUDA，也可能是 CPU，以首个代码单元的输出为准。

In [ ]:
# 基线训练配置
model = CNN().to(device)
loss_func = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
epochs = 3

print(f"模型设备: {next(model.parameters()).device}")
print(f"训练轮数: {epochs}")

In [ ]:
# 基线训练循环
print("开始训练...")
for epoch in range(epochs):
    model.train()
    train_loss = 0
    correct = 0
    total = 0
    start_time = time.time()
    
    for batch_idx, (imgs, labels) in enumerate(train_loader):
        # 将数据移到当前设备
        imgs = imgs.to(device)
        labels = labels.to(device)
        
        # 前向传播
        outputs = model(imgs)
        loss = loss_func(outputs, labels)
        
        # 反向传播与优化
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # 统计
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    epoch_time = time.time() - start_time
    train_loss /= len(train_loader)
    accuracy = 100. * correct / total
    
    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Loss: {train_loss:.4f} "
          f"Accuracy: {accuracy:.2f}% "
          f"Time: {epoch_time:.2f}s")

print("\n基线训练完成")

### 3. 自动迁移到 NPU

导入 `torch_npu` 和 `transfer_to_npu` 后，常见 CUDA API 会映射到 NPU API。下面列出本实验涉及的主要映射：

| CUDA API | NPU 映射 |
| --- | --- |
| `torch.cuda.is_available()` | `torch.npu.is_available()` |
| `.cuda()` | `.npu()` |
| `torch.device('cuda:0')` | `torch.device('npu:0')` |
| `torch.cuda.amp` | `torch.npu.amp` |

下面的代码保留原有设备选择和训练循环，只增加自动迁移所需的导入。

In [ ]:
# 自动迁移：导入以下两个模块
import torch_npu
from torch_npu.contrib import transfer_to_npu  # 导入后自动映射 CUDA API

# 保留基线代码中的设备选择方式
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")
print("(transfer_to_npu 已将 CUDA API 映射到 NPU)")

# 沿用相同的模型定义
model = CNN().to(device)
loss_func = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

print(f"\n模型设备: {next(model.parameters()).device}")

In [ ]:
# 沿用基线训练循环
epochs = 3
print("NPU 上开始训练（自动迁移模式）...")

for epoch in range(epochs):
    model.train()
    train_loss = 0
    correct = 0
    total = 0
    start_time = time.time()
    
    for batch_idx, (imgs, labels) in enumerate(train_loader):
        # .to(device) 自动映射到 NPU
        imgs = imgs.to(device)
        labels = labels.to(device)
        
        # 前向传播
        outputs = model(imgs)
        loss = loss_func(outputs, labels)
        
        # 反向传播与优化
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # 统计
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    epoch_time = time.time() - start_time
    train_loss /= len(train_loader)
    accuracy = 100. * correct / total
    
    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Loss: {train_loss:.4f} "
          f"Accuracy: {accuracy:.2f}% "
          f"Time: {epoch_time:.2f}s")

print("\nNPU 训练完成")
print(f"模型运行在: {next(model.parameters()).device}")

自动迁移适合快速确认代码能否在 NPU 上运行。复杂项目、第三方库或工具未覆盖的接口仍需逐项检查，必要时改用 NPU API。如果算子未在 NPU 上实现，框架可能回落到 CPU 执行；日志和性能数据可以帮助发现这类回退。判断迁移是否生效时，以模型参数所在设备的输出为准。

### 4. 配置 AMP 混合精度训练

`amp.autocast()` 包裹前向计算，由框架决定各运算使用 FP16 还是 FP32。`GradScaler` 缩放损失并更新缩放因子，减少半精度训练中的梯度下溢。

下面重新创建模型、损失函数和优化器，再使用 NPU AMP 运行训练。

In [ ]:
import torch
import torch.nn as nn
import torch_npu
from torch_npu.npu import amp               # NPU 版 AMP 模块
from torch_npu.contrib import transfer_to_npu

device = torch.device('cuda:0')  # 自动迁移会映射为 npu:0

# 模型、损失函数、优化器
model = CNN().to(device)
loss_func = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# ===== 新增：创建 GradScaler =====
scaler = amp.GradScaler()

epochs = 3
print(f"AMP 混合精度训练配置完成")
print(f"设备: {device}")

In [ ]:
# AMP 训练循环
print("开始 AMP 混合精度训练...")

for epoch in range(epochs):
    model.train()
    train_loss = 0
    correct = 0
    total = 0
    start_time = time.time()
    
    for batch_idx, (imgs, labels) in enumerate(train_loader):
        imgs = imgs.to(device)
        labels = labels.to(device)
        
        # ===== 修改：使用 autocast 包裹前向计算 =====
        with amp.autocast():
            outputs = model(imgs)
            loss = loss_func(outputs, labels)
        
        optimizer.zero_grad()
        
        # ===== 修改：使用 scaler.scale(loss).backward() =====
        scaler.scale(loss).backward()
        
        # ===== 修改：使用 scaler.step(optimizer) =====
        scaler.step(optimizer)
        
        # ===== 修改：更新 scaler =====
        scaler.update()
        
        # 统计
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    epoch_time = time.time() - start_time
    train_loss /= len(train_loader)
    accuracy = 100. * correct / total
    
    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Loss: {train_loss:.4f} "
          f"Accuracy: {accuracy:.2f}% "
          f"Time: {epoch_time:.2f}s")

print("\nAMP 混合精度训练完成")

与普通训练循环相比，AMP 增加了以下调用：

| 位置 | 普通训练 | NPU AMP |
| --- | --- | --- |
| 前向计算 | 直接计算 | `with amp.autocast():` |
| 反向传播 | `loss.backward()` | `scaler.scale(loss).backward()` |
| 参数更新 | `optimizer.step()` | `scaler.step(optimizer)`，随后调用 `scaler.update()` |

昇腾 NPU 不支持 `DataParallel`。多卡训练应使用 `DistributedDataParallel`，并为每个进程绑定对应的 NPU。

### 5. 测试并保存模型

将模型切换到评估模式，在测试集上计算损失和准确率，然后保存模型、优化器状态及本次准确率。

In [ ]:
# 测试评估
model.eval()
test_loss = 0
correct = 0
total = 0

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)
        outputs = model(imgs)
        loss = loss_func(outputs, labels)
        
        test_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

test_loss /= len(test_loader)
accuracy = 100. * correct / total

print(f"测试集 Loss: {test_loss:.4f}")
print(f"测试集 Accuracy: {accuracy:.2f}%")
print(f"正确预测数: {correct}/{total}")

In [ ]:
# 保存模型权重
torch.save({
    'epoch': epochs,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'accuracy': accuracy,
}, 'mnist_cnn_npu.pth.tar')

print("模型已保存为 mnist_cnn_npu.pth.tar")
import os
print(f"模型大小: {os.path.getsize('mnist_cnn_npu.pth.tar') / 1024 / 1024:.2f} MB")

## 实验总结

`torch_npu` 可以把常见 CUDA 风格的 PyTorch 代码映射到昇腾 NPU，AMP 则通过 `autocast` 和 `GradScaler` 调整训练精度。自动迁移有明确的覆盖范围，复杂项目仍需检查第三方接口和算子支持情况；多卡训练应采用 DDP。